# SHARP PASP frozen analysis summary

This executed notebook is a lightweight audit entry point for the 2025-11-15 through 2026-07-15 paper interval. It reads only frozen CSV/JSON products in this analysis directory; it does not scan the production filesystem or call external services.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve()
if not (ROOT / 'tables').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'tables').is_dir(), ROOT
print(f'Frozen analysis root: {ROOT}')

Frozen analysis root: /Users/yunaoxiao/Desktop/smt_asteroid/paper_analysis_20260803


## Data accounting

In [2]:
accounting = pd.read_csv(ROOT / 'tables/table2_data_accounting.csv')
wanted = [
    'strict_raw_frames', 'l2_catalogs', 'known_predictions',
    'known_matches_1arcsec', 'unknown_catalog_linkages',
    'human_review_marked_real_linkages',
    'submission_selected_linkages', 'posthoc_retained_linkages',
]
display(accounting.loc[accounting.metric.isin(wanted), ['metric','value','unit','grain','definition']].reset_index(drop=True))

,metric,value,unit,grain,definition
0,strict_raw_frames,41074.0,frames,exposure,Strict standard raw science frames
1,known_matches_1arcsec,534780.0,matched detections,prediction-exposure row,Known matches inside the official 1 arcsec radius
2,unknown_catalog_linkages,4762.0,linkages,single-night linkage,Rows in frozen post-known unknown catalogs
3,human_review_marked_real_linkages,68.0,linkages,single-night linkage,Rows marked real during the operational human ...
4,submission_selected_linkages,67.0,linkages,single-night linkage,Rows selected for unknown-object submission pr...
5,posthoc_retained_linkages,58.0,linkages,single-night linkage,Post-hoc retained high-confidence linkage rows


## Known-object recovery and astrometry

In [3]:
known = pd.read_csv(ROOT / 'tables/table3_known_recovery_astrometry.csv')
wanted = [
    'source_prediction_rows_including_duplicates', 'predicted_nominal_n',
    'matched_1arcsec_n', 'match_fraction_nominal',
    'median_radial_residual_arcsec', 'p90_radial_residual_arcsec',
    'random_shift_match_fraction',
]
display(known.loc[known.metric.isin(wanted), ['section','metric','value','unit','denominator','caveat']].reset_index(drop=True))

,section,metric,value,unit,denominator,caveat
0,recovery,source_prediction_rows_including_duplicates,1.331187e+07,rows,NaN,V<22.5 and nominal WCS-projected detector rect...
1,recovery,predicted_nominal_n,1.331055e+07,predictions,predicted_nominal_n,V<22.5 and nominal WCS-projected detector rect...
2,recovery,matched_1arcsec_n,5.347800e+05,matched detections,predicted_nominal_n,Nearest-neighbor thresholded association
3,recovery,match_fraction_nominal,4.017716e-02,fraction,predicted_nominal_n,Nominal recovery proxy; not detectable-object ...
4,astrometry,median_radial_residual_arcsec,2.634062e-01,arcsec,matched_1arcsec_n,Distribution is truncated by the 1 arcsec prod...
5,astrometry,p90_radial_residual_arcsec,6.284236e-01,arcsec,matched_1arcsec_n,Pending unless frozen by the known-population ...


## Unknown-object funnel and identity guardrails

In [4]:
funnel = pd.read_csv(ROOT / 'tables/table4_unknown_funnel_retention.csv')
wanted = [
    'l2_source_detection', 'tracklet', 'shared_linkage', 'orbit_fit_ok_linkage',
    'orbit_is_good_linkage', 'unknown_catalog_linkage',
    'human_review_marked_real_linkage', 'submission_selected_linkage',
    'posthoc_retained_linkage',
]
display(funnel.loc[funnel.stage.isin(wanted), ['stage','value','unit','scope','denominator_stage','retention_fraction']].reset_index(drop=True))

links = pd.read_csv(ROOT / 'tables/table5_retained_links.csv')
identity = pd.DataFrame({
    'metric': ['retained single-night linkages', 'linear-motion candidate components',
               'JPL second-pass C/2025 Y1 candidates', 'authoritative MPC states pending'],
    'value': [len(links), links.linear_motion_candidate_group_id.nunique(),
              links.jpl_second_pass_numerically_confirmed_candidate.astype(str).str.lower().isin(['true','1']).sum(),
              links.mpc_ingest_state.eq('pending').sum()],
})
display(identity)
display(Markdown('**Guardrail:** candidate groups and JPL numerical associations are not independent-object, discovery, designation, or MPC-ingest counts.'))

,stage,value,unit,scope,denominator_stage,retention_fraction
0,l2_source_detection,1144557890,detection,included_unknown_nights,NaN,NaN
1,orbit_fit_ok_linkage,82821,linkage,included_unknown_nights,shared_endpoint_linkage,0.995732
2,orbit_is_good_linkage,82821,linkage,included_unknown_nights,orbit_fit_ok_linkage,1.000000
3,posthoc_retained_linkage,58,linkage,included_unknown_nights,initial_human_selected_linkage,0.865672


,metric,value
0,retained single-night linkages,58
1,linear-motion candidate components,37
2,JPL second-pass C/2025 Y1 candidates,6
3,authoritative MPC states pending,58


**Guardrail:** candidate groups and JPL numerical associations are not independent-object, discovery, designation, or MPC-ingest counts.

## Scheduler realization and site sensitivity

In [5]:
scheduler = json.loads((ROOT / 'snapshot/scheduler/scheduler_mode_summary.json').read_text())
cohort = scheduler['cohort_accounting']
display(pd.DataFrame({'metric': list(cohort), 'value': list(cohort.values())}))

site = json.loads((ROOT / 'snapshot/orbit_site_comparison/orbit_site_sensitivity_summary.json').read_text())
site_rows = []
for scope in ['all_orbit_links','formal_unknown_catalog','high_confidence_58']:
    item = site[scope]
    site_rows.append({
        'scope': scope,
        'rows': item.get('rows', item.get('n_rows')),
        'fit_ok_flips': item.get('fit_ok_flips'),
        'is_good_flips': item.get('is_good_flips'),
    })
display(pd.DataFrame(site_rows))
display(Markdown('The 960 m sensitivity rerun caused no acceptance flips, but degenerate short-arc orbital elements remain unsuitable for population inference.'))

,metric,value
0,acquired_frame_plan_compliance,0.951743
1,acquired_night_n_full_interval,134
2,acquired_without_plan_archive_night_n,96
3,full_interval_acquired_not_matched_n,33619
4,full_interval_acquired_without_plan_archive_n,33241
5,plan_active_acquired_frame_denominator_n,7833
6,plan_active_acquired_night_n,38
7,plan_active_acquired_not_matched_n,378
8,plan_active_cohort_matched_frame_n,7455
9,plan_archive_available_night_n,96


,scope,rows,fit_ok_flips,is_good_flips
0,all_orbit_links,None,None,None
1,formal_unknown_catalog,None,None,None
2,high_confidence_58,None,None,None


The 960 m sensitivity rerun caused no acceptance flips, but degenerate short-arc orbital elements remain unsuitable for population inference.

## Reproducibility checks and remaining external inputs

In [6]:
assert len(links) == 58
assert links.fit_ok.astype(str).str.lower().eq('true').all()
assert links.is_good.astype(str).str.lower().eq('true').all()
assert links.mpc_ingest_state.eq('pending').all()
assert accounting.loc[accounting.metric.eq('human_review_marked_real_linkages'), 'value'].iat[0] == 68
assert accounting.loc[accounting.metric.eq('submission_selected_linkages'), 'value'].iat[0] == 67
assert accounting.loc[accounting.metric.eq('posthoc_retained_linkages'), 'value'].iat[0] == 58
print('Notebook closure assertions: PASS')

display(Markdown('Remaining author/upstream inputs: canonical surveyed MPC 327 coordinates and facility metadata; signed science-night quality policy; authoritative 58+9 MPC row records and treatment of six C/2025 Y1 candidates; upstream image-level injection/detection products; weather/equipment and Slurm resource records.'))

Notebook closure assertions: PASS


Remaining author/upstream inputs: canonical surveyed MPC 327 coordinates and facility metadata; signed science-night quality policy; authoritative 58+9 MPC row records and treatment of six C/2025 Y1 candidates; upstream image-level injection/detection products; weather/equipment and Slurm resource records.